## import libraries

In [5]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pickle
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam, RMSprop, SGD



## Paths Setup

In [6]:
BASE_DIR = Path.cwd().parent.parent
DATA_DIR     = BASE_DIR / "data"
FEATURES_DIR = DATA_DIR / "dl_features"
MODELS_DIR   = FEATURES_DIR

print("BASE_DIR      :", BASE_DIR)
print("FEATURES_DIR  :", FEATURES_DIR)

BASE_DIR      : c:\Users\mereh\nlp-fake-news-detector-transformers
FEATURES_DIR  : c:\Users\mereh\nlp-fake-news-detector-transformers\data\dl_features


## Load Preprocessed Features

In [7]:
def load_features():
    print("Loading preprocessed features...")

    X_train = np.load(FEATURES_DIR / "X_train_pad.npy")
    X_val   = np.load(FEATURES_DIR / "X_val_pad.npy")
    X_test  = np.load(FEATURES_DIR / "X_test_pad.npy")

    y_train = np.load(FEATURES_DIR / "y_train.npy")
    y_val   = np.load(FEATURES_DIR / "y_val.npy")
    y_test  = np.load(FEATURES_DIR / "y_test.npy")

    with open(FEATURES_DIR / "meta.pkl", "rb") as f:
        meta = pickle.load(f)

    with open(FEATURES_DIR / "tokenizer.pkl", "rb") as f:
        tokenizer = pickle.load(f)

    print(f"  X_train shape : {X_train.shape}")
    print(f"  X_val   shape : {X_val.shape}")
    print(f"  X_test  shape : {X_test.shape}")
    print(f"  max_len       : {meta['max_len']}")
    print(f"  max_words     : {meta['max_words']}")
    print(f"  Vocab size    : {len(tokenizer.word_index)}")

    return X_train, X_val, X_test, y_train, y_val, y_test, meta, tokenizer

In [8]:
X_train, X_val, X_test, y_train, y_val, y_test, meta, tokenizer = load_features()

Loading preprocessed features...
  X_train shape : (963423, 20)
  X_val   shape : (222329, 20)
  X_test  shape : (296438, 20)
  max_len       : 20
  max_words     : 20000
  Vocab size    : 295251


## Load GloVe Embeddings

In [2]:
import gensim.downloader as api

def load_glove_gensim():
    print("Loading GloVe (gensim)...")
    return api.load("glove-wiki-gigaword-100")


def load_fasttext_gensim():
    print("Loading FastText (gensim)...")
    return api.load("fasttext-wiki-news-subwords-300")


## Build Embedding Matrix

In [10]:
def build_embedding_matrix_gensim(tokenizer, model, max_words, embedding_dim):
    embedding_matrix = np.zeros((max_words, embedding_dim))

    for word, i in tokenizer.word_index.items():
        if i >= max_words:
            continue

        try:
            embedding_matrix[i] = model[word]
        except:
            continue

    return embedding_matrix

## Build CNN Model with Pretrained Embeddings

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

def build_model_with_pretrained(max_words, max_len, embedding_dim, embedding_matrix, optimizer, trainable = False):

    model = Sequential([
        Input(shape=(max_len,)),

        Embedding(
            input_dim=max_words,
            output_dim=embedding_dim,
            weights=[embedding_matrix],
            trainable= trainable
        ),

        Conv1D(128, 5, activation='relu'),
        GlobalMaxPooling1D(),

        Dense(64, activation='relu'),
        Dropout(0.5),

        Dense(1, activation='sigmoid')
    ])

    model.compile(
        loss='binary_crossentropy',
        optimizer=optimizer,
        metrics=['accuracy']
    )

    return model

## Training + Evaluation

In [12]:
def train_and_evaluate(model, name, X_train, y_train, X_val, y_val, X_test, y_test):
    print(f"\nTraining {name}...")

    early_stop = EarlyStopping(patience=3, restore_best_weights=True)

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=5,
        batch_size=128,
        callbacks=[early_stop],
        verbose=1
    )

    preds = (model.predict(X_test) > 0.5).astype(int)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    print(f"\n{name} Results:")
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1 Score :", f1)

    return acc,prec, rec, f1

## Optimizer Selector

In [13]:
import tensorflow as tf

def get_optimizer(name):
    if name == "Adam":
        return tf.keras.optimizers.Adam(learning_rate=0.001)

    elif name == "RMSprop":
        return tf.keras.optimizers.RMSprop(learning_rate=0.001)

    elif name == "SGD":
        return tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)

In [10]:
glove_model = load_glove_gensim()

Loading GloVe (gensim)...


In [3]:
fasttext_model = load_fasttext_gensim()

Loading FastText (gensim)...
[==================================================] 100.0% 958.5/958.4MB downloaded


## Frozen Experiments FastText

In [ ]:
max_words = meta['max_words']
max_len   = meta['max_len']

results_fasttext_frozen = {}
embedding_dim_fasttext = 300


embedding_matrix_fasttext = build_embedding_matrix_gensim(
    tokenizer, fasttext_model, max_words, embedding_dim_fasttext
)

for opt_name in ["Adam", "RMSprop", "SGD"]:
    opt = get_optimizer(opt_name)

    print(f"\n== FastText_{opt_name}_frozen ===\n")

    model = build_model_with_pretrained(
        max_words,
        max_len,
        embedding_dim_fasttext,
        embedding_matrix_fasttext,
        optimizer=opt,
        trainable=False
    )

    results_fasttext_frozen[f"FastText_{opt_name}_frozen"] = train_and_evaluate(
        model,
        f"FastText_{opt_name}_frozen",
        X_train, y_train,
        X_val, y_val,
        X_test, y_test
    )


=== FastText_Adam_frozen ===


Training FastText_Adam_frozen...
Epoch 1/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 108s 14ms/step - accuracy: 0.7702 - loss: 0.4825 - val_accuracy: 0.7807 - val_loss: 0.4612
Epoch 2/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 121s 16ms/step - accuracy: 0.7889 - loss: 0.4528 - val_accuracy: 0.7855 - val_loss: 0.4537
Epoch 3/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 111s 15ms/step - accuracy: 0.7967 - loss: 0.4390 - val_accuracy: 0.7858 - val_loss: 0.4515
Epoch 4/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 115s 15ms/step - accuracy: 0.8022 - loss: 0.4283 - val_accuracy: 0.7875 - val_loss: 0.4504
Epoch 5/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 96s 13ms/step - accuracy: 0.8077 - loss: 0.4187 - val_accuracy: 0.7869 - val_loss: 0.4542
9264/9264 ━━━━━━━━━━━━━━━━━━━━ 19s 2ms/step

FastText_Adam_frozen Results:
Accuracy : 0.7881411964727869
Precision: 0.7826051725302181
Recall   : 0.7910719770656292
F1 Score : 0.7868157979599111

=== FastText_RMSprop_frozen ===


Training FastText_RMSprop_frozen...
Epoch 1/5


## Frozen Experiments GloVe

In [ ]:
max_words = meta['max_words']
max_len   = meta['max_len']

results_GloVe_frozen = {}
embedding_dim_glove = 100

embedding_matrix_glove = build_embedding_matrix_gensim(
    tokenizer, glove_model, max_words, embedding_dim_glove
)

for opt_name in ["Adam", "RMSprop", "SGD"]:

    opt = get_optimizer(opt_name)

    print(f"\n=== GloVe_{opt_name}_frozen ===\n")

    model = build_model_with_pretrained(
        max_words,
        max_len,
        embedding_dim_glove,
        embedding_matrix_glove,
        optimizer=opt,
        trainable=False
    )

    results_GloVe_frozen[f"GloVe_{opt_name}_frozen"] = train_and_evaluate(
        model,
        f"GloVe + {opt_name}",
        X_train, y_train,
        X_val, y_val,
        X_test, y_test
    )

Loading preprocessed features...
  X_train shape : (963423, 20)
  X_val   shape : (222329, 20)
  X_test  shape : (296438, 20)
  max_len       : 20
  max_words     : 20000
  Vocab size    : 295251

=== GloVe_Adam_frozen ===


Training GloVe + Adam...
Epoch 1/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 35s 5ms/step - accuracy: 0.7463 - loss: 0.5162 - val_accuracy: 0.7632 - val_loss: 0.4879
Epoch 2/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 35s 5ms/step - accuracy: 0.7689 - loss: 0.4835 - val_accuracy: 0.7692 - val_loss: 0.4796
Epoch 3/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 35s 5ms/step - accuracy: 0.7770 - loss: 0.4711 - val_accuracy: 0.7706 - val_loss: 0.4765
Epoch 4/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 35s 5ms/step - accuracy: 0.7821 - loss: 0.4620 - val_accuracy: 0.7717 - val_loss: 0.4765
Epoch 5/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - accuracy: 0.7857 - loss: 0.4549 - val_accuracy: 0.7734 - val_loss: 0.4743
9264/9264 ━━━━━━━━━━━━━━━━━━━━ 10s 1ms/step

GloVe + Adam Results:
Accuracy : 0.7730891451163481
Pre

## Fine-tuned Experiments FastText

In [15]:
max_words = meta['max_words']
max_len   = meta['max_len']

results_fasttext_finetuned = {}
embedding_dim_fasttext = 300


embedding_matrix_fasttext = build_embedding_matrix_gensim(
    tokenizer, fasttext_model, max_words, embedding_dim_fasttext
)

for opt_name in ["Adam", "RMSprop", "SGD"]:
    opt = get_optimizer(opt_name)

    print(f"\n=== FastText_{opt_name}_finetuned ===\n")

    model = build_model_with_pretrained(
        max_words,
        max_len,
        embedding_dim_fasttext,
        embedding_matrix_fasttext,
        optimizer=opt,
        trainable=True
    )

    results_fasttext_finetuned[f"FastText_{opt_name}_finetuned"] = train_and_evaluate(
        model,
        f"FastText_{opt_name}_finetuned",
        X_train, y_train,
        X_val, y_val,
        X_test, y_test
    )


=== FastText_Adam_finetuned ===


Training FastText_Adam_finetuned...
Epoch 1/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 487s 64ms/step - accuracy: 0.7870 - loss: 0.4572 - val_accuracy: 0.7955 - val_loss: 0.4373
Epoch 2/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 521s 69ms/step - accuracy: 0.8143 - loss: 0.4091 - val_accuracy: 0.7974 - val_loss: 0.4346
Epoch 3/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 506s 67ms/step - accuracy: 0.8414 - loss: 0.3571 - val_accuracy: 0.7909 - val_loss: 0.4697
Epoch 4/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 319s 42ms/step - accuracy: 0.8706 - loss: 0.2957 - val_accuracy: 0.7842 - val_loss: 0.5160
Epoch 5/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 347s 46ms/step - accuracy: 0.8952 - loss: 0.2414 - val_accuracy: 0.7774 - val_loss: 0.6381
9264/9264 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step

FastText_Adam_finetuned Results:
Accuracy : 0.7972662074362936
Precision: 0.8000555613431954
Recall   : 0.7862939831405071
F1 Score : 0.7931150814141623

=== FastText_RMSprop_finetuned ===


Training FastText_RMSprop_finetun

## Fine-tuned Experiments GloVe

In [ ]:
max_words = meta['max_words']
max_len   = meta['max_len']

results_GloVe_finetuned = {}
embedding_dim_glove = 100

embedding_matrix_glove = build_embedding_matrix_gensim(
    tokenizer, glove_model, max_words, embedding_dim_glove
)

for opt_name in ["Adam", "RMSprop", "SGD"]:
    opt = get_optimizer(opt_name)
    
    print(f"\n=== GloVe_{opt_name}_finetuned ===\n")

    model2 = build_model_with_pretrained(
        max_words,
        max_len,
        embedding_dim_glove,
        embedding_matrix_glove,
        optimizer=opt,
        trainable=True
    )

    results_GloVe_finetuned[f"GloVe_{opt_name}_finetuned"] = train_and_evaluate(
        model2,
        f"GloVe_{opt_name}_finetuned",
        X_train, y_train,
        X_val, y_val,
        X_test, y_test
    )

Loading preprocessed features...
  X_train shape : (963423, 20)
  X_val   shape : (222329, 20)
  X_test  shape : (296438, 20)
  max_len       : 20
  max_words     : 20000
  Vocab size    : 295251

=== GloVe_Adam_finetuned ===


Training GloVe_Adam_finetuned...
Epoch 1/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 91s 12ms/step - accuracy: 0.7797 - loss: 0.4682 - val_accuracy: 0.7940 - val_loss: 0.4411
Epoch 2/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 90s 12ms/step - accuracy: 0.8047 - loss: 0.4271 - val_accuracy: 0.7964 - val_loss: 0.4367
Epoch 3/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 87s 12ms/step - accuracy: 0.8173 - loss: 0.4029 - val_accuracy: 0.7965 - val_loss: 0.4375
Epoch 4/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 93s 12ms/step - accuracy: 0.8321 - loss: 0.3734 - val_accuracy: 0.7934 - val_loss: 0.4663
Epoch 5/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 88s 12ms/step - accuracy: 0.8488 - loss: 0.3393 - val_accuracy: 0.7853 - val_loss: 0.4845
9264/9264 ━━━━━━━━━━━━━━━━━━━━ 10s 1ms/step

GloVe_Adam_finetuned Results:
Accuracy 

In [ ]:
results_GloVe_frozen

{'GloVe_Adam_frozen': (0.7730891451163481,
  0.7723697959632624,
  0.7668816763932972,
  0.7696159523785059),
 'GloVe_RMSprop_frozen': (0.7720096613794453,
  0.7554210628519645,
  0.7965939729019488,
  0.7754613864017674),
 'GloVe_SGD_frozen': (0.771051619562944,
  0.7601497988566589,
  0.7841780144022388,
  0.7719769789780305)}

In [ ]:
results_GloVe_finetuned

{'GloVe_Adam_finetuned': (0.7954546987903035,
  0.8130240153978507,
  0.76117538650558,
  0.7862458445994776),
 'GloVe_RMSprop_finetuned': (0.7953096431631572,
  0.7933754913689968,
  0.7921299614347633,
  0.7927522371746704),
 'GloVe_SGD_finetuned': (0.7818295899985832,
  0.778641913932947,
  0.7804170506126071,
  0.7795284716919385)}

In [16]:
results_fasttext_frozen

{'FastText_Adam_frozen': (0.7881411964727869,
  0.7826051725302181,
  0.7910719770656292,
  0.7868157979599111),
 'FastText_RMSprop_frozen': (0.788572989967548,
  0.7857892870779468,
  0.7866420941264803,
  0.7862154593425635),
 'FastText_SGD_frozen': (0.7869065369487043,
  0.7981994102997166,
  0.7612982492065117,
  0.7793122482418415)}

In [17]:
results_fasttext_finetuned

{'FastText_Adam_finetuned': (0.7972662074362936,
  0.8000555613431954,
  0.7862939831405071,
  0.7931150814141623),
 'FastText_RMSprop_finetuned': (0.7982579831195731,
  0.789225072555626,
  0.8074331934063684,
  0.7982253112453187),
 'FastText_SGD_finetuned': (0.7971548856759255,
  0.795756803769295,
  0.7931333401590389,
  0.7944429061249048)}

## Comparison Table

In [ ]:
import pandas as pd

comparison = []

for name, res in results_GloVe_frozen.items():
    acc, prec, rec, f1 = res 

    comparison.append({
        "model": name,
        "type": "Frozen",
        "accuracy": acc,
        "f1": f1
    })

for name, res in results_GloVe_finetuned.items():
    acc, prec, rec, f1 = res

    comparison.append({
        "model": name,
        "type": "Fine-tuned",
        "accuracy": acc,
        "f1": f1
    })

comparison.append({
    "model": "Baseline_CNN",
    "type": "Baseline",
    "accuracy": 0.7935,
    "f1": 0.7949
})

df_GloVe = pd.DataFrame(comparison)

display(df_GloVe)

,model,type,accuracy,f1
0,GloVe_Adam_frozen,Frozen,0.773089,0.769616
1,GloVe_RMSprop_frozen,Frozen,0.772010,0.775461
2,GloVe_SGD_frozen,Frozen,0.771052,0.771977
3,GloVe_Adam_finetuned,Fine-tuned,0.795455,0.786246
4,GloVe_RMSprop_finetuned,Fine-tuned,0.795310,0.792752
5,GloVe_SGD_finetuned,Fine-tuned,0.781830,0.779528
6,Baseline_CNN,Baseline,0.793500,0.794900


In [18]:
import pandas as pd

comparison_fasttext = []

for name, res in results_fasttext_frozen.items():
    acc, prec, rec, f1 = res 

    comparison_fasttext.append({
        "model": name,
        "type": "Frozen",
        "accuracy": acc,
        "f1": f1
    })

for name, res in results_fasttext_finetuned.items():
    acc, prec, rec, f1 = res

    comparison_fasttext.append({
        "model": name,
        "type": "Fine-tuned",
        "accuracy": acc,
        "f1": f1
    })
comparison_fasttext.append({
    "model": "Baseline_CNN",
    "type": "Baseline",
    "accuracy": 0.7935,
    "f1": 0.7949
})    

df_fasttext = pd.DataFrame(comparison_fasttext)

display(df_fasttext)

,model,type,accuracy,f1
0,FastText_Adam_frozen,Frozen,0.788141,0.786816
1,FastText_RMSprop_frozen,Frozen,0.788573,0.786215
2,FastText_SGD_frozen,Frozen,0.786907,0.779312
3,FastText_Adam_finetuned,Fine-tuned,0.797266,0.793115
4,FastText_RMSprop_finetuned,Fine-tuned,0.798258,0.798225
5,FastText_SGD_finetuned,Fine-tuned,0.797155,0.794443
6,Baseline_CNN,Baseline,0.793500,0.794900


## Best GloVe Model Selection

In [ ]:
best_GloVe_model = df_GloVe.loc[df_GloVe["accuracy"].idxmax()]

print("Best Model (by Accuracy)")
print(best_GloVe_model)

=== Best Model (by Accuracy) ===
model       GloVe_Adam_finetuned
type                  Fine-tuned
accuracy                0.795455
f1                      0.786246
Name: 3, dtype: object


## Best FastText Model Selection

In [19]:
best_fasttext = df_fasttext.loc[df_fasttext["accuracy"].idxmax()]

print("Best Model (by Accuracy)")
print(best_fasttext)

Best Model (by Accuracy)
model       FastText_RMSprop_finetuned
type                        Fine-tuned
accuracy                      0.798258
f1                            0.798225
Name: 4, dtype: object
